byte_tracker.py

In [46]:
import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image
from collections import defaultdict
import time
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from types import SimpleNamespace

In [47]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [48]:
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Define the transform (same as during training)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and modify the detection model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
# model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model_Perfect.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()


SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            )
          )
        )
        (2): Invert

Helper Functions

In [49]:
def xyxy_to_xywh(xyxy):
    """Convert bounding box from [x1, y1, x2, y2] to [cx, cy, w, h]."""
    x1, y1, x2, y2 = xyxy[:4]
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1
    return np.array([cx, cy, w, h], dtype=xyxy.dtype)

def predict(image_input, model, device, threshold=0.2, nms_threshold=0.2, max_detections=1):
    """
    Predict detections on an image using the SSDLite model.
    
    Args:
        image_input (str or np.ndarray): File path or OpenCV BGR image.
        model: The detection model.
        device: Computation device.
        threshold (float): Confidence threshold.
        nms_threshold (float): (Currently unused) IoU threshold for NMS.
        max_detections (int): Maximum number of top scoring detections to return.
        
    Returns:
        orig_img (np.ndarray): Original image (BGR).
        detections (np.ndarray): Array of detections in [x1, y1, x2, y2, score] format.
        inference_time (float): Inference time in milliseconds.
    """
    from PIL import Image
    import torchvision.transforms as T

    # Define transform (should match training)
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    # Convert input to PIL image and keep a copy in BGR for display
    if isinstance(image_input, np.ndarray):
        pil_img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
        orig_img = image_input.copy()
    elif isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
        orig_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Unsupported image input type.")

    img_tensor = transform(pil_img).to(device)
    img_tensor = img_tensor.unsqueeze(0)

    start = cv2.getTickCount()
    with torch.no_grad():
        outputs = model(img_tensor)
    end = cv2.getTickCount()
    inference_time = (end - start) / cv2.getTickFrequency() * 1000.0
    # print(f"Inference time: {inference_time:.1f} ms")

    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()   # shape: (N, 4)
    scores = output['scores'].cpu().numpy()   # shape: (N,)

    # Filter detections below threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]

    if len(boxes) > 0:
        # Get indices that would sort scores in descending order
        sorted_indices = np.argsort(scores)[::-1]
        # Select the top `max_detections` indices
        top_indices = sorted_indices[:max_detections]
        boxes = boxes[top_indices]
        scores = scores[top_indices]
    else:
        detections = np.empty((0, 5), dtype=np.float32)
        return orig_img, detections, inference_time

    detections = np.hstack((boxes, scores.reshape(-1, 1)))
    return orig_img, detections, inference_time



In [50]:
def create_dummy_results(dets):
    """
    Given dets (an array of shape (N, 5) with [x1, y1, x2, y2, score]),
    return a dummy results object with attributes:
      - xywh: bounding boxes in [cx, cy, w, h] format
      - conf: confidence scores
      - cls: class labels (assumed to be 0 for all detections)
    """
    if len(dets) == 0:
        return SimpleNamespace(
            xywh=np.empty((0, 4), dtype=np.float32),
            conf=np.empty((0,), dtype=np.float32),
            cls=np.empty((0,), dtype=np.int32)
        )
    boxes_xywh = np.array([xyxy_to_xywh(det[:4]) for det in dets])
    conf = dets[:, 4]
    cls = np.zeros_like(conf, dtype=np.int32)
    return SimpleNamespace(xywh=boxes_xywh, conf=conf, cls=cls)

 Tracker Setup
 Here's a breakdown of each parameter in your DummyArgs class and how it affects the tracker:

track_thresh (0.2):
This is the minimum detection confidence required for a detection to be considered for tracking. Detections with a score below this value will be ignored in the tracking process. In your case, only detections with confidence ≥ 0.2 are used for initializing or updating tracks.

track_buffer (40):
This parameter defines the maximum number of frames a track can remain “lost” (i.e. not updated with a new detection) before it is removed from the active track list. A higher value allows tracks to be maintained longer even if an object is temporarily occluded or missed.

match_thresh (0.8):
This is the threshold for matching a new detection to an existing track—often based on Intersection over Union (IoU) or a similar cost metric. If the cost between a track's predicted position and a detection is below this threshold (or equivalently, if the similarity is above a certain level), the detection is considered a match for that track.

mot20 (False):
This flag indicates whether to use MOT20-specific settings. MOT20 is a benchmark for multi-object tracking with specific challenges. When set to True, the tracker might adjust certain thresholds or processing steps to suit MOT20 scenarios.

track_low_thresh (0.1):
This is a lower confidence threshold used during a secondary association step. Detections with confidence above 0.1 (but below the high threshold) might be used to update tracks that weren't matched with high-confidence detections, giving a chance to recover missed objects.

track_high_thresh (0.2):
This is the high confidence threshold used in the primary association step. Detections with confidence above 0.2 are considered high-confidence and are prioritized for matching with existing tracks.

new_track_thresh (0.2):
This threshold determines whether a detection is confident enough to initialize a new track. If a detection's score is below this value, even if it doesn't match any existing track, it might not be used to start a new track.

fuse_score (False):
When enabled (set to True), this option instructs the tracker to combine (or “fuse”) the detection score with the matching cost during data association. Fusing the score can sometimes improve association by giving additional weight to high-confidence detections when matching them to existing tracks.


In ByteTrack (and similar multi‐object tracking pipelines), detections are typically matched to existing tracks in two passes, often referred to as:
- Primary (first) association step
- Secondary (second) association step

Primary (first) association
- Goal: Confidently match your best (high‐confidence) detections to existing tracks.
- Detections: Only those above a relatively higher confidence threshold (track_high_thresh).
- Why: By restricting this pass to high‐confidence detections, you reduce the risk of false‐positive matches. Tracks that get matched in this step are considered “activated” or “updated” with minimal uncertainty.

Secondary (second) association
- Goal: Recover leftover (unmatched) tracks by using leftover (unmatched) detections that have moderate confidence.
- Detections: Those that are above a lower threshold (track_low_thresh), but below the high threshold used in the primary step.
- Why: Sometimes an object is detected with less confidence. If we skip these moderate detections entirely, we may fail to update tracks that were not matched in the first pass. By giving leftover tracks a second chance to match lower‐confidence detections, we can recover objects that might otherwise be lost or incorrectly marked as removed.

In [51]:
class DummyArgs:
    track_thresh = 0.15        # Minimum detection confidence for tracking
    track_buffer = 60         # Maximum frames to keep a lost track
    match_thresh = 0.9       # Threshold for matching (e.g., IoU)
    # mot20 = False             # MOT20-specific settings flag
    track_low_thresh = 0.15    # Low threshold for detections (second association)
    track_high_thresh = 0.2   # High threshold for detections (first association)
    new_track_thresh = .9   # Threshold to initialize a new track
    fuse_score = True        # Whether to fuse detection score
    # min_box_area: 70  # threshold for min box areas(for tracker evaluation, not used for now)

args = DummyArgs()

In [ ]:
import cv2
import numpy as np
import random
import time
from collections import defaultdict
# Import BYTETracker from your trackers package
from trackers.byte_tracker import BYTETracker
tracker = BYTETracker(args=args, frame_rate=30)
# Assuming predict, create_dummy_results, and BYTETracker (tracker) are already defined and imported.

# video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\TestFile_video.mp4"
# video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Baseline.mp4"
# video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\3_Mice.mp4"

# Global dictionary to store colors for each track_id
track_colors = {}

def get_color_for_id(track_id):
    """
    Returns a BGR color tuple for the given track_id.
    If the track_id is new, a random color is assigned and stored.
    """
    if track_id not in track_colors:
        # Generate a random BGR color
        color = (random.randint(0, 255), 
                 random.randint(0, 255), 
                 random.randint(0, 255))
        track_colors[track_id] = color
    return track_colors[track_id]


# Example usage with ByteTrack
# -----------------------------------------------------------------------------
# from trackers.byte_tracker import BYTETracker
# tracker = BYTETracker(args=args, frame_rate=30)
# predict, create_dummy_results, model, device also assumed to be defined

k_max = 3
cap = cv2.VideoCapture(video_path)
cv2.namedWindow("Tracking", cv2.WINDOW_NORMAL)

# Dictionary to store track history for drawing tracking lines: track_id -> list of (cx, cy)
track_history = defaultdict(list)
start_time_global = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    orig_img, detections, inf_time = predict(frame, model, device, threshold=0.3, max_detections=k_max)
    
    # Convert detections to ByteTrack-compatible format
    results = create_dummy_results(detections)
    # Update tracker
    tracks = tracker.update(results, img=None)
    # Annotate detections with bounding boxes (optional)
    def plot_detections(orig_img, dummy_results):
        annotated_img = orig_img.copy()
        for box, score in zip(dummy_results.xywh, dummy_results.conf):
            cx, cy, w, h = box
            x1 = int(cx - w / 2)
            y1 = int(cy - h / 2)
            x2 = int(cx + w / 2)
            y2 = int(cy + h / 2)
            # Here we use a fixed color for detections, but you could also randomize or unify it with track color
            # cv2.rectangle(annotated_img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        return annotated_img

    annotated_frame = plot_detections(orig_img, results)

    # Draw tracking polylines, IDs, and confidence
    if tracks.size > 0:
        for track in tracks:
            x1, y1, x2, y2, tid, score, cls, idx = track
            tid = int(tid)
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2

            # Get a color for this ID
            color = get_color_for_id(tid)

            # Store history for polylines
            track_history[tid].append((cx, cy))
            if len(track_history[tid]) > 30:
                track_history[tid].pop(0)
            pts = np.array(track_history[tid], dtype=np.int32).reshape(-1, 1, 2)

            # Draw the polylines for the track
            cv2.polylines(annotated_frame, [pts], isClosed=False, color=color, thickness=2)

            # Draw a bounding box for the tracked object using the same color
            x1_i, y1_i, x2_i, y2_i = map(int, [x1, y1, x2, y2])
            cv2.rectangle(annotated_frame, (x1_i, y1_i), (x2_i, y2_i), color, 2)

            # Label with track ID and confidence
            label = f"ID:{tid}, Conf: {score:.2f}"
            cv2.putText(annotated_frame, label, (x1_i, y1_i - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # Display inference time
    cv2.putText(annotated_frame, f"Inference: {inf_time:.1f} ms", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow("Tracking", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting display loop.")
        break

cap.release()
cv2.destroyAllWindows()
print("Video processing complete.")


# HEatMap

In [ ]:
import cv2
import numpy as np
import time
from collections import defaultdict

# -----------------------------------------------------
# Main script: track rat, build permanent bounding-box heatmap using a kernel patch
# -----------------------------------------------------


video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
cap = cv2.VideoCapture(video_path)
cv2.namedWindow("Heatmap", cv2.WINDOW_NORMAL)

# Read first frame to get dimensions and initialize heatmap accumulator
ret, frame = cap.read()
if not ret:
    raise ValueError("Error reading the first frame from video.")

h, w = frame.shape[:2]
heatmap_acc = np.zeros((h, w), dtype=np.float32)

# No decay => positions remain permanent
decay_factor = 1.0

# Fixed parameters for heatmap accumulation:
max_visits = 50   # Maximum "visit" count for full saturation (red)
increment = 1     # Increment added for each detection pass

# Define a kernel patch: a block covering the current pixel and 10 pixels around it.
# This creates a 21x21 patch of ones multiplied by the increment.
kernel_radius = 10
kernel_size = 2 * kernel_radius + 1
kernel = np.ones((kernel_size, kernel_size), dtype=np.float32) * increment

# Create a custom LUT for mapping intensities from green to red:
#   0   -> Green (BGR: [0, 255, 0])
#   255 -> Red   (BGR: [0, 0, 255])
lut = np.zeros((256, 1, 3), dtype=np.uint8)
for i in range(256):
    fraction = i / 255.0
    blue = 0
    green = int(255 * (1 - fraction))
    red = int(255 * fraction)
    lut[i, 0] = [blue, green, red]

# Optional: reset video to the start
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 1) Run detection and tracking (detect rats in the frame)
    orig_img, detections, inf_time = predict(frame, model, device, threshold=0.2, max_detections=1)

    # 2) Convert detections to ByteTrack-compatible format
    results = create_dummy_results(detections)

    # 3) Update ByteTrack tracker
    tracks = tracker.update(results, img=None)

    # 4) For each tracked rat, update the heatmap accumulator with a kernel patch
    if tracks.size > 0:
        for track in tracks:
            # Assuming track format: [x1, y1, x2, y2, tid, score, cls, idx]
            x1, y1, x2, y2, tid, score, cls, idx = track
            # Compute the centroid
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # Determine the region in the heatmap to update using the kernel.
            x_start = max(cx - kernel_radius, 0)
            x_end = min(cx + kernel_radius + 1, w)  # +1 because slicing is exclusive
            y_start = max(cy - kernel_radius, 0)
            y_end = min(cy + kernel_radius + 1, h)

            # Compute corresponding kernel region in case we're near image boundaries
            kx_start = 0 if cx - kernel_radius >= 0 else kernel_radius - cx
            ky_start = 0 if cy - kernel_radius >= 0 else kernel_radius - cy
            kx_end = kx_start + (x_end - x_start)
            ky_end = ky_start + (y_end - y_start)

            # Add the kernel patch to the heatmap accumulator
            heatmap_acc[y_start:y_end, x_start:x_end] += kernel[ky_start:ky_end, kx_start:kx_end]

    # 5) Apply no decay so the accumulated heatmap remains permanent
    heatmap_acc *= decay_factor  # decay_factor is 1.0

    # 6) Clip the accumulator values to max_visits
    acc_limited = np.clip(heatmap_acc, 0, max_visits)

    # 7) Scale [0, max_visits] to [0, 255]
    heatmap_norm = (acc_limited / max_visits) * 255.0
    heatmap_norm = heatmap_norm.astype(np.uint8)

    # 8) Convert single-channel heatmap to a 3-channel image
    heatmap_bgr = cv2.cvtColor(heatmap_norm, cv2.COLOR_GRAY2BGR)

    # 9) Apply the custom LUT to map grayscale intensities to green-to-red colors
    heatmap_color = cv2.LUT(heatmap_bgr, lut)

    # 10) Create a mask for nonzero (hot) pixels
    mask = heatmap_norm > 0

    # 11) Blend the custom heatmap only onto the hot areas of the original image
    temp_blend = cv2.addWeighted(orig_img, 0.3, heatmap_color, 0.7, 0)
    overlay = orig_img.copy()
    overlay[mask] = temp_blend[mask]

    # 12) Optionally display inference time on the overlay
    cv2.putText(overlay, f"Inference: {inf_time:.1f} ms", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # 13) Show the result
    cv2.imshow("Heatmap", overlay)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting display loop.")
        break

cap.release()
cv2.destroyAllWindows()
print("Video processing complete.")


Exiting display loop.
Video processing complete.
